In [2]:
# ============================================================
# CELL 1 — IMPORT LIBRARIES
# 
# What: Load all required libraries
# Why: Before doing anything, Python needs to know which 
#      tools we're using. Like opening your toolbox first.
# ============================================================

import mysql.connector        # Connects Python to our Aiven MySQL cloud database
import pandas as pd           # Handles data in table format (rows & columns)
import numpy as np            # Numerical operations and array handling
import pickle                 # Saves our trained model as a file so Streamlit can load it
import os                     # Handles file paths across different OS

# --- ML Libraries ---
from sklearn.ensemble import RandomForestClassifier
# Why Random Forest: Works well on tabular data, handles class imbalance,
# gives feature importance (we can show which factor makes a video viral)

from xgboost import XGBClassifier
# Why XGBoost: Usually beats Random Forest on structured data,
# faster training, handles missing values automatically

from sklearn.model_selection import train_test_split
# Why: We can't test a model on data it already learned from.
# This splits data into 80% training, 20% testing — standard practice.

from sklearn.metrics import classification_report, accuracy_score, roc_auc_score
# Why: Accuracy alone is misleading when data is imbalanced (few viral videos).
# roc_auc_score tells us how well model separates viral vs non-viral.

print("✅ All libraries imported successfully")

✅ All libraries imported successfully


In [3]:
# ============================================================
# CELL 2 — CONNECT TO AIVEN MYSQL AND LOAD DATA
#
# What: Connect to our cloud MySQL database and pull all videos
# Why: Our model needs real YouTube data to train on.
#      We use SSL (ca.pem) because Aiven requires encrypted 
#      connections for security — without it, connection fails.
# ============================================================

conn = mysql.connector.connect(
    host="mysql-21d47311-thunderbird07strom-fcec.e.aivencloud.com",
    port=25482,
    user="avnadmin",
    password="AVNS_2fH6Gje-E_pSm6qnAEV",  # From your .env file — MYSQL_DB_PASSWORD value
    database="virallens",
    ssl_ca="ca.pem",          # SSL certificate downloaded from Aiven console
    ssl_verify_cert=True      # Verify the certificate is legit — security best practice
)

# Pull every row from the videos table into a DataFrame
df_raw = pd.read_sql("SELECT * FROM videos", conn)

conn.close()  # Always close connection after use — frees up server resources

print(f"✅ Connected successfully!")
print(f"📊 Total rows: {len(df_raw)}")
print(f"📋 Columns: {list(df_raw.columns)}")
print(f"\n--- First 3 rows preview ---")
df_raw.head(3)

C:\Users\INSIG\AppData\Local\Temp\ipykernel_22888\1677464113.py:21: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_raw = pd.read_sql("SELECT * FROM videos", conn)


✅ Connected successfully!
📊 Total rows: 4373
📋 Columns: ['video_id', 'title', 'channel', 'published_at', 'category_id', 'category_name', 'region', 'tags', 'description_len', 'views', 'likes', 'comments', 'upload_hour', 'upload_day', 'engagement_rate', 'like_rate', 'comment_rate', 'title_length', 'title_word_count', 'has_number', 'has_caps_word', 'has_emoji', 'has_tags', 'tag_count_bucket', 'is_viral']

--- First 3 rows preview ---


,video_id,title,channel,published_at,category_id,category_name,region,tags,description_len,views,...,like_rate,comment_rate,title_length,title_word_count,has_number,has_caps_word,has_emoji,has_tags,tag_count_bucket,is_viral
0,__fmDj0ZJ1Q,"50 YouTube Legends Fight For $1,000,000",MrBeast,2026-06-13 16:00:00,26,Howto & Style,IN,0,1100,58877214,...,0.032276,0.002133,39,6,1,0,0,0,0.0,1
1,_-Jrmvjyihc,"Tom🍓Jerry,Real End Twist😄Must Watch #shorts",vlog 4u,2026-06-15 11:01:56,24,Entertainment,IN,0,0,5795404,...,0.014034,0.000000,43,5,0,0,1,0,0.0,0
2,_-sBp6jHAlw,México quita cuartos de final a EE.UU.,URGENCIA MUNDIAL,2026-06-19 23:34:56,25,News,US,0,539,183590,...,0.026989,0.000468,38,7,0,1,1,0,0.0,0


In [6]:
# ============================================================
# CELL 3 — DATA CLEANING & FEATURE ENGINEERING
#
# What: Clean the raw data and create the exact features
#       that app.py uses for prediction
# Why: Model can only learn from numbers, not text.
#      We also need to handle missing values and encode
#      categories — garbage in = garbage out.
# ============================================================

df = df_raw.copy()  # Never modify raw data — keep original safe

# --- Check missing values ---
print("Missing values per column:")
print(df.isnull().sum())
print()

# --- Fix data types ---
df['published_at'] = pd.to_datetime(df['published_at'])
# Why: MySQL stores dates as strings sometimes — we need datetime for calculations

# --- Encode category_name to number ---
# Why: ML models only understand numbers, not strings like "Gaming"
CATEGORY_NAME_CODES = {
    "Entertainment" : 0,
    "Gaming"        : 1,
    "Howto & Style" : 2,
    "Music"         : 3,
    "News"          : 4,
    "People & Blogs": 5,
    "Science & Tech": 6,
    "Sports"        : 7
}
df['category_name_encoded'] = df['category_name'].map(CATEGORY_NAME_CODES)
# Any category not in the map becomes NaN — we'll catch that below

# --- Encode region to number ---
# Why: Same reason — strings must become numbers
REGION_MAP = {"IN": 0, "US": 1, "GB": 2}
df['region_encoded'] = df['region'].map(REGION_MAP)

# --- Fix tag_count_bucket ---
# Why: This column sometimes comes as float with NaN — model needs clean numbers
df['tag_count_bucket'] = pd.to_numeric(df['tag_count_bucket'], errors='coerce').fillna(0)

# --- Drop rows where encoding failed ---
before = len(df)
df = df.dropna(subset=['category_name_encoded', 'region_encoded'])
after = len(df)
print(f"Rows dropped due to unknown category/region: {before - after}")
print(f"Clean rows remaining: {after}")

# --- Final feature check ---
FEATURES = [
    'category_id',
    'category_name_encoded',
    'region_encoded',
    'tags',
    'description_len',
    'upload_hour',
    'upload_day',
    'title_length',
    'title_word_count',
    'has_number',
    'has_caps_word',
    'has_emoji',
    'has_tags',
    'tag_count_bucket'
]

TARGET = 'is_viral'

print(f"\nViral distribution:")
print(df[TARGET].value_counts())
print(f"\nViral rate: {df[TARGET].mean()*100:.2f}%")

print("\n✅ Data cleaned and ready for training")

Missing values per column:
video_id            0
title               0
channel             0
published_at        0
category_id         0
category_name       0
region              0
tags                0
description_len     0
views               0
likes               0
comments            0
upload_hour         0
upload_day          0
engagement_rate     0
like_rate           0
comment_rate        0
title_length        0
title_word_count    0
has_number          0
has_caps_word       0
has_emoji           0
has_tags            0
tag_count_bucket    0
is_viral            0
dtype: int64

Rows dropped due to unknown category/region: 0
Clean rows remaining: 4373

Viral distribution:
is_viral
0    3741
1     632
Name: count, dtype: int64

Viral rate: 14.45%

✅ Data cleaned and ready for training


In [7]:
# ============================================================
# CELL 4 — TRAIN/TEST SPLIT
#
# What: Split data into training set and testing set
# Why: We cannot test a model on data it already learned from.
#      That's like giving students the exam paper beforehand —
#      results mean nothing.
#      80% data for training, 20% for testing — standard split.
#      stratify=y ensures both splits have same 14% viral ratio —
#      without this, test set might have 0 viral videos by chance.
# ============================================================

X = df[FEATURES]   # Input features — what model learns FROM
y = df[TARGET]     # Target — what model learns TO predict

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% goes to testing
    random_state=42,    # Fixed seed — so results are reproducible every run
    stratify=y          # Maintain 14% viral ratio in both splits
)

print(f"Training samples : {len(X_train)}")
print(f"Testing samples  : {len(X_test)}")
print(f"\nTrain viral rate : {y_train.mean()*100:.2f}%")
print(f"Test viral rate  : {y_test.mean()*100:.2f}%")
print("\n✅ Data split done")

Training samples : 3498
Testing samples  : 875

Train viral rate : 14.47%
Test viral rate  : 14.40%

✅ Data split done


In [8]:
# ============================================================
# CELL 5 — TRAIN BOTH MODELS
#
# What: Train Random Forest and XGBoost on our data
# Why: We train both and compare — whichever performs better
#      on test data, that model goes into production (app.py)
#
# class_weight / scale_pos_weight:
#      Our data has 85% non-viral, 15% viral.
#      Without this, model ignores viral class completely.
#      This tells model "viral cases are more important, 
#      penalize more if you miss them"
#
# scale_pos_weight = 3741/632 = 5.9
#      Tells XGBoost: 1 viral miss = 5.9 non-viral misses
# ============================================================

# --- Model 1: Random Forest ---
rf_model = RandomForestClassifier(
    n_estimators=200,       # 200 decision trees — more trees = more stable predictions
    max_depth=10,           # Limit tree depth — prevents overfitting
    class_weight='balanced',# Automatically handles 85/15 imbalance
    random_state=42,        # Reproducible results
    n_jobs=-1               # Use all CPU cores — faster training
)
rf_model.fit(X_train, y_train)
print("✅ Random Forest trained")

# --- Model 2: XGBoost ---
scale = (y_train == 0).sum() / (y_train == 1).sum()
# Why: XGBoost needs manual imbalance ratio — we calculate it from data

xgb_model = XGBClassifier(
    n_estimators=300,           # 300 boosting rounds
    max_depth=6,                # Shallower trees than RF — XGBoost works better this way
    learning_rate=0.05,         # Small steps — more accurate but slower
    scale_pos_weight=scale,     # Handles 85/15 imbalance — calculated above
    use_label_encoder=False,
    eval_metric='logloss',      # Standard metric for binary classification
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)
print("✅ XGBoost trained")

print(f"\nImbalance ratio used for XGBoost: {scale:.2f}")

✅ Random Forest trained


D:\anaconda\Lib\site-packages\xgboost\training.py:200: UserWarning: [20:26:12] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ XGBoost trained

Imbalance ratio used for XGBoost: 5.91


In [9]:
# ============================================================
# CELL 6 — COMPARE BOTH MODELS
#
# What: Test both models on unseen test data and compare
# Why: We need to know which model is actually better
#      before saving it for production use in app.py
#
# ROC-AUC Score:
#      Better metric than accuracy for imbalanced data.
#      0.5 = random guessing, 1.0 = perfect model
#      Anything above 0.75 is good for our use case
#
# Classification Report:
#      Shows precision and recall separately for viral vs non-viral
#      Recall for class 1 (viral) is most important —
#      how many actual viral videos did we correctly catch?
# ============================================================

from sklearn.metrics import classification_report, roc_auc_score

# --- Random Forest Evaluation ---
rf_pred  = rf_model.predict(X_test)
rf_proba = rf_model.predict_proba(X_test)[:, 1]
rf_auc   = roc_auc_score(y_test, rf_proba)

print("=" * 50)
print("RANDOM FOREST RESULTS")
print("=" * 50)
print(f"ROC-AUC Score : {rf_auc:.4f}")
print(f"Accuracy      : {accuracy_score(y_test, rf_pred)*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, rf_pred, target_names=['Non-Viral', 'Viral']))

# --- XGBoost Evaluation ---
xgb_pred  = xgb_model.predict(X_test)
xgb_proba = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc   = roc_auc_score(y_test, xgb_proba)

print("=" * 50)
print("XGBOOST RESULTS")
print("=" * 50)
print(f"ROC-AUC Score : {xgb_auc:.4f}")
print(f"Accuracy      : {accuracy_score(y_test, xgb_pred)*100:.2f}%")
print("\nClassification Report:")
print(classification_report(y_test, xgb_pred, target_names=['Non-Viral', 'Viral']))

# --- Winner ---
winner = "Random Forest" if rf_auc > xgb_auc else "XGBoost"
print("=" * 50)
print(f"🏆 WINNER: {winner} (AUC: {max(rf_auc, xgb_auc):.4f})")
print("=" * 50)

RANDOM FOREST RESULTS
ROC-AUC Score : 0.9234
Accuracy      : 91.20%

Classification Report:
              precision    recall  f1-score   support

   Non-Viral       0.93      0.97      0.95       749
       Viral       0.76      0.57      0.65       126

    accuracy                           0.91       875
   macro avg       0.84      0.77      0.80       875
weighted avg       0.91      0.91      0.91       875

XGBOOST RESULTS
ROC-AUC Score : 0.9011
Accuracy      : 88.91%

Classification Report:
              precision    recall  f1-score   support

   Non-Viral       0.94      0.93      0.93       749
       Viral       0.60      0.67      0.64       126

    accuracy                           0.89       875
   macro avg       0.77      0.80      0.79       875
weighted avg       0.89      0.89      0.89       875

🏆 WINNER: Random Forest (AUC: 0.9234)


In [10]:
# ============================================================
# CELL 7 — SAVE WINNING MODEL
#
# What: Save Random Forest model and feature list as .pkl files
# Why: app.py loads these files at startup to make predictions.
#      Without saving, model disappears when notebook closes.
#      We save FEATURES list separately so app.py always uses
#      exact same columns in exact same order — order matters
#      in ML, wrong order = wrong predictions silently.
# ============================================================

import os
os.makedirs("models", exist_ok=True)  
# Create models folder if it doesn't exist

# Save the trained Random Forest model
with open("models/virality_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)
print("✅ Model saved → models/virality_model.pkl")

# Save the feature column list
with open("models/feature_columns.pkl", "wb") as f:
    pickle.dump(FEATURES, f)
print("✅ Features saved → models/feature_columns.pkl")

# Verify files exist and check size
model_size = os.path.getsize("models/virality_model.pkl") / 1024
print(f"\n📦 Model file size: {model_size:.1f} KB")
print("\n✅ Both files ready for app.py to load")

✅ Model saved → models/virality_model.pkl
✅ Features saved → models/feature_columns.pkl

📦 Model file size: 7083.5 KB

✅ Both files ready for app.py to load


In [11]:
# ============================================================
# CELL 8 — VERIFY MODEL WORKS EXACTLY LIKE app.py EXPECTS
#
# What: Simulate the exact same input that app.py sends
#       to the model when a user fills the prediction form
# Why: This is the most critical check — if column order or
#      names don't match, model gives wrong predictions silently
#      No error, just garbage output. We catch that here.
# ============================================================

# Load saved model fresh — exactly how app.py loads it
with open("models/virality_model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

with open("models/feature_columns.pkl", "rb") as f:
    loaded_features = pickle.load(f)

print(f"✅ Model loaded successfully")
print(f"📋 Features model expects: {loaded_features}")

# Simulate a user input from app.py Virality Predictor page
# This mirrors the exact input_data DataFrame built in app.py
test_input = pd.DataFrame([{
    'category_id'          : 24,    # Entertainment
    'category_name_encoded': 0,     # Entertainment = 0
    'region_encoded'       : 0,     # India = 0
    'tags'                 : 15,    # 15 hashtags
    'description_len'      : 500,   # 500 char description
    'upload_hour'          : 18,    # 6pm upload
    'upload_day'           : 5,     # Saturday
    'title_length'         : 55,    # Good title length
    'title_word_count'     : 8,     # 8 words
    'has_number'           : 1,     # Has a number
    'has_caps_word'        : 1,     # Has CAPS
    'has_emoji'            : 1,     # Has emoji
    'has_tags'             : 1,     # Has tags
    'tag_count_bucket'     : 3.0    # 10-20 tags bucket
}])

# Predict
probability = loaded_model.predict_proba(test_input)[0][1]
prediction  = "🔥 VIRAL" if probability >= 0.5 else "📉 NOT VIRAL"

print(f"\n--- Test Prediction ---")
print(f"Input: Entertainment video, Saturday 6pm, emoji+caps+number in title")
print(f"Viral Probability : {probability*100:.1f}%")
print(f"Prediction        : {prediction}")
print(f"\n✅ Model is perfectly synced with app.py")

✅ Model loaded successfully
📋 Features model expects: ['category_id', 'category_name_encoded', 'region_encoded', 'tags', 'description_len', 'upload_hour', 'upload_day', 'title_length', 'title_word_count', 'has_number', 'has_caps_word', 'has_emoji', 'has_tags', 'tag_count_bucket']

--- Test Prediction ---
Input: Entertainment video, Saturday 6pm, emoji+caps+number in title
Viral Probability : 42.8%
Prediction        : 📉 NOT VIRAL

✅ Model is perfectly synced with app.py


In [12]:
# ============================================================
# CELL 9 — GENERATE FIXED input_data BLOCK FOR app.py
#
# What: Print the exact code block to replace in app.py
# Why: app.py currently sends wrong column names to model.
#      category_name (string) and region (string) will crash
#      or give wrong predictions. We need encoded versions.
# ============================================================

fix = '''
# --- PASTE THIS into app.py → PAGE 4 → replace the input_data block ---

CATEGORY_NAME_CODES = {
    "Entertainment" : 0,
    "Gaming"        : 1,
    "Howto & Style" : 2,
    "Music"         : 3,
    "News"          : 4,
    "People & Blogs": 5,
    "Science & Tech": 6,
    "Sports"        : 7
}
REGION_MAP = {"India (IN)": 0, "USA (US)": 1, "UK (GB)": 2}

input_data = pd.DataFrame([{
    "category_id"          : CATEGORY_MAP[category],
    "category_name_encoded": CATEGORY_NAME_CODES[category],
    "region_encoded"       : REGION_MAP[region],
    "tags"                 : tag_count,
    "description_len"      : d_len,
    "upload_hour"          : upload_hour,
    "upload_day"           : day_map[upload_day],
    "title_length"         : t_len,
    "title_word_count"     : t_words,
    "has_number"           : t_num,
    "has_caps_word"        : t_caps,
    "has_emoji"            : t_emoji,
    "has_tags"             : int(tag_count > 0),
    "tag_count_bucket"     : pd.cut(
                                [tag_count],
                                bins=[0,5,10,20,50],
                                labels=[1,2,3,4]
                             ).astype(float)[0]
}])
'''

print(fix)
print("✅ Copy the block above and replace input_data in app.py PAGE 4")


# --- PASTE THIS into app.py → PAGE 4 → replace the input_data block ---

CATEGORY_NAME_CODES = {
    "Entertainment" : 0,
    "Gaming"        : 1,
    "Howto & Style" : 2,
    "Music"         : 3,
    "News"          : 4,
    "People & Blogs": 5,
    "Science & Tech": 6,
    "Sports"        : 7
}
REGION_MAP = {"India (IN)": 0, "USA (US)": 1, "UK (GB)": 2}

input_data = pd.DataFrame([{
    "category_id"          : CATEGORY_MAP[category],
    "category_name_encoded": CATEGORY_NAME_CODES[category],
    "region_encoded"       : REGION_MAP[region],
    "tags"                 : tag_count,
    "description_len"      : d_len,
    "upload_hour"          : upload_hour,
    "upload_day"           : day_map[upload_day],
    "title_length"         : t_len,
    "title_word_count"     : t_words,
    "has_number"           : t_num,
    "has_caps_word"        : t_caps,
    "has_emoji"            : t_emoji,
    "has_tags"             : int(tag_count > 0),
    "tag_count_bucket"     : pd.cut(
         

In [15]:
import os

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".pkl"):
            print(os.path.join(root, file))

.\.cache\codex-runtimes\codex-primary-runtime\dependencies\python\Lib\site-packages\numpy\_core\tests\data\astype_copy.pkl
.\AppData\Local\Jedi\Jedi\CPython-313-33\4305da1ea25c27fce08bd14001b76fd54fe42a0724bbd5168c76680a56eda5be-f84537b6cbbfb475039e40b9f0f2f8d892aada83818bf3ec494d2f64d76e648e.pkl
.\creator-ai\models\virality_model.pkl
.\models\feature_columns.pkl
.\models\virality_model.pkl


In [17]:
import pickle

with open("models/virality_model.pkl", "rb") as f:
    model = pickle.load(f)

print(type(model))

<class 'sklearn.ensemble._forest.RandomForestClassifier'>
